# Burfisher 3e — TBL94: regulación de la producción de MFG en USA

La regulación reduce un 1% la producción de MFG de la actividad MFG de USA. El modelo
calcula el impuesto a la producción que logra ese mismo resultado: el **impuesto
sombra** de la regulación (Tabla 9.4).

En GEMPACK (`TBL94.EXP`) el cierre es `swap qca("MFG","MFG","USA") = to("MFG","MFG","USA")`
y el shock `qca = -1`. Aquí se arma modificando el `ShockBlock` **sólo en este
notebook** con `@overwrite`.

In [ ]:
from pathlib import Path

from pyomo.environ import value

from equilibria._local_refs import nus333_dir
from equilibria.blocks.gtap import ShockBlock, overwrite
from equilibria.templates.gtap import GTAPParameters
from equilibria.templates.gtap.gtap_block_model import build_block_model
from equilibria.templates.gtap.gtap_contract import GTAPClosureConfig
from equilibria.templates.gtap.gtap_multiperiod_driver import solve_multiperiod
from equilibria.templates.gtap.shocks import apply_shock

## 1. Datos: la base nus333 (2 regiones × 3 sectores) y el `.prm` del ejercicio

In [ ]:
har: Path = nus333_dir()
p = GTAPParameters()
p.load_from_har(
    basedata_path=har / "basedata.har",
    sets_path=har / "sets.har",
    default_path=har / "default.prm",
    baserate_path=har / "baserate.har",
)

## 2. El cierre de la regulación, con `@overwrite`

- `prdtx_rai[USA,MFG,MFG]`, el impuesto a la producción, deja de ser un instrumento
  fijo y pasa a ser endógeno;
- `b.target` declara el objetivo `qca_target` sobre la producción `x`: una variable
  nueva, fija en 1, y la fila `x[shock] = qca_target × x[check]`.

Se libera una incógnita y se agrega una fila, así que el sistema sigue cuadrado. Con
`period="shock"` el cierre cambia **sólo en el shock**, como el `swap` de GEMPACK: en
base y check el impuesto queda fijo y el check replica el benchmark.

`domains=("r", "a", "i")` va explícito porque la celda `("USA", "MFG", "MFG")` es
ambigua: MFG es a la vez actividad y bien. Los hooks quedan registrados en la clase y
duran todo el notebook; si re-ejecutas la celda, el hook se reemplaza, no se duplica.

In [ ]:
CELL = ("USA", "MFG", "MFG")


@overwrite(ShockBlock, period="shock")
def regulation(b):
    b.endogenous("prdtx_rai", CELL)
    b.target(
        "qca_target",
        CELL,
        quantity=lambda m, r, a, i: m.x[r, a, i],
        domains=("r", "a", "i"),
    )

## 3. Cierre global y modelo

`build_block_model` arma los 3 períodos (base / check / shock) y ya incluye los hooks
de arriba.

In [ ]:
gc = GTAPClosureConfig(
    name="base",
    closure_type="MCP",
    capital_mobility="sluggish",
    fix_endowments=False,
    fix_taxes=False,
    fix_technology=False,
    if_sub=False,
    savf_flag="capFlex",
    numeraire="pnum",
)
m, _ = build_block_model(p, p.sets, gc, "ROW", base_calibrated=False, ref_gdx=None)
for t in ("check", "shock"):
    print(f"prdtx_rai[USA,MFG,MFG] libre en {t}:", not m.prdtx_rai[(*CELL, t)].fixed)
print("fila eq_qca_target:", list(m.eq_qca_target))

## 4. El shock: el objetivo pasa a 0,99 × la producción de check

In [ ]:
apply_shock(m, {"qca_target": {CELL: -1.0}})
print(
    f"qca_target{CELL}: check {value(m.qca_target[(*CELL, 'check')]):.6f}"
    f" -> shock {value(m.qca_target[(*CELL, 'shock')]):.6f}"
)

## 5. Resolver check y shock

In [ ]:
res = solve_multiperiod(
    m,
    p,
    gc,
    ref_gdx=None,
    skip_base_solve=True,  # GAMS no resuelve la base
    mute_welfare=True,
    seed_from_prior=False,
    mode="gtap",
    solve_check=True,
)
print("codes:", {k: int(res[k]["code"]) for k in res})

## 6. Resultados (% de check a shock) contra GAMS y GEMPACK

In [ ]:
def pct(var, *k):
    comp = getattr(m, var)
    return 100.0 * (value(comp[(*k, "shock")]) / value(comp[(*k, "check")]) - 1.0)


t_chk = value(m.prdtx_rai[(*CELL, "check")])
t_shk = value(m.prdtx_rai[(*CELL, "shock")])
filas = [
    ("impuesto sombra (potencia 1+to)", 100 * ((1 + t_shk) / (1 + t_chk) - 1), 2.249523, 2.2500),
    ("producción MFG de MFG USA (x)", pct("x", *CELL), -1.0, -1.0),
    ("producción AGR USA (xp)", pct("xp", "USA", "AGR"), 1.366686, 1.3671),
    ("producción MFG USA (xp)", pct("xp", "USA", "MFG"), -1.0, -1.0),
    ("producción SER USA (xp)", pct("xp", "USA", "SER"), 0.189215, 0.1892),
]
print(f"{'':34s}{'equilibria':>12s}{'GAMS':>12s}{'GEMPACK':>12s}")
for nombre, eq, gams, gem in filas:
    print(f"{nombre:34s}{eq:+12.4f}{gams:+12.4f}{gem:+12.4f}")
print(f"tasa to: {100 * t_chk:.4f}% -> {100 * t_shk:.4f}%")

equilibria y GAMS coinciden en todas las celdas. GEMPACK difiere de los dos en hasta
0,0005pp en estas cifras; corre este `.EXP` con Gragg 2-4-6, pocos pasos de
extrapolación.